# NefroQuest — Revisão de Questões (gpt-5.4-mini)

**4 passos:**
1. Cole sua OpenAI API key e rode a **Célula 1**
2. Faça upload dos 3 arquivos e rode a **Célula 2**
3. *(Só se quiser re-processar tudo do zero)* Rode a **Célula 3** para apagar lotes antigos
4. Rode a **Célula 4** e aguarde — ao final o `index.html` revisado será baixado automaticamente

In [ ]:
# ══════════════════════════════════════════════════════════
# CÉLULA 1 — API Key
# ══════════════════════════════════════════════════════════
OPENAI_API_KEY = "sk-..."  # ← substitua pela sua chave

import os
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
!pip install -q openai
print("✓ Pronto. Rode a Célula 2.")

In [ ]:
# ══════════════════════════════════════════════════════════
# CÉLULA 2 — Upload dos arquivos
# Necessários: index.html · review_questions.py · apply_reviews.py
# ══════════════════════════════════════════════════════════
from google.colab import files
import re

print("Selecione os 3 arquivos: index.html, review_questions.py, apply_reviews.py")
uploaded = files.upload()

needed  = {"index.html", "review_questions.py", "apply_reviews.py"}
missing = needed - set(uploaded.keys())
if missing:
    print(f"\n⚠ Faltando: {missing} — rode esta célula novamente.")
else:
    with open("index.html", encoding="utf-8") as f:
        html = f.read()
    bloco = html[html.find("const topics = ["):html.find("\n];", html.find("const topics = ["))]
    n = len(re.findall(r'"t":', bloco))
    print(f"\n✓ Uploads OK. Questões encontradas: {n}")
    print("Rode a Célula 3 (para apagar lotes antigos) ou vá direto para a Célula 4.")

In [ ]:
# ══════════════════════════════════════════════════════════
# CÉLULA 3 — Apagar lotes antigos (opcional)
# Rode APENAS se quiser re-processar tudo do zero.
# Para continuar de onde parou, PULE esta célula.
# ══════════════════════════════════════════════════════════
import os, glob

jsons = sorted(glob.glob("reviewed_*.json"))
if not jsons:
    print("Nenhum lote encontrado — nada a apagar.")
else:
    for f in jsons:
        os.remove(f)
        print(f"  Deletado: {f}")
    print(f"\n✓ {len(jsons)} lote(s) apagado(s). A Célula 4 vai re-processar tudo.")

In [ ]:
# ══════════════════════════════════════════════════════════
# CÉLULA 4 — Revisão completa
# Modelo: gpt-5.4-mini · escalonamento automático para gpt-5.4
# Tempo estimado: 60–90 min para 1034 questões
# Se desconectar: rode novamente — lotes prontos são pulados.
# ══════════════════════════════════════════════════════════
import subprocess, json, os, shutil
from google.colab import files

TOTAL    = 1034
BATCH_SZ = 100

batches = []
s = 0
while s < TOTAL:
    batches.append((s, min(s + BATCH_SZ, TOTAL)))
    s += BATCH_SZ

total_ok  = 0
total_err = 0

for i, (s, e) in enumerate(batches):
    out = f"reviewed_{s}_{e}.json"

    if os.path.exists(out):
        with open(out) as f_:
            data = json.load(f_)
        ok = sum(1 for r in data if not r.get("_error"))
        print(f"[Lote {i+1:02d}/{len(batches)}] {s}–{e}: JÁ FEITO ({ok}/{e-s} OK) — pulando")
        total_ok  += ok
        total_err += (e - s) - ok
        continue

    print(f"\n[Lote {i+1:02d}/{len(batches)}] Revisando questões {s}–{e-1}...")
    r = subprocess.run(
        ["python", "review_questions.py",
         "--start", str(s), "--end", str(e), "--out", out]
    )

    if r.returncode != 0 or not os.path.exists(out):
        print(f"  ERRO no lote {i+1}. Verifique a API key e rode novamente.")
        break

    with open(out) as f_:
        data = json.load(f_)
    ok  = sum(1 for r_ in data if not r_.get("_error"))
    err = len(data) - ok
    total_ok  += ok
    total_err += err

    subprocess.run(["python", "apply_reviews.py", out])
    shutil.copy("index.html", f"index_backup_lote{i+1:02d}.html")
    print(f"  → {ok} revisadas OK | {err} com erro")

print(f"""
════════════════════════════════
CONCLUÍDO
  Revisadas com sucesso : {total_ok}
  Com erro de API       : {total_err}
  index.html final      : {os.path.getsize('index.html'):,} bytes
════════════════════════════════
Baixando index.html...
""")
files.download("index.html")

In [ ]:
# ══════════════════════════════════════════════════════════
# CÉLULA 5 — Retentar questões com erro (opcional)
# Rode apenas se a Célula 4 reportou erros de API.
# ══════════════════════════════════════════════════════════
import json, os, time, importlib.util, subprocess
from openai import OpenAI

spec = importlib.util.spec_from_file_location("rq", "review_questions.py")
rq   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(rq)

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
all_q  = rq.extract_questions("index.html")

TOTAL, BATCH_SZ = 1034, 100
batches = [(s, min(s + BATCH_SZ, TOTAL)) for s in range(0, TOTAL, BATCH_SZ)]

for s, e in batches:
    fname = f"reviewed_{s}_{e}.json"
    if not os.path.exists(fname):
        continue
    with open(fname) as f:
        data = json.load(f)
    errs = [r for r in data if r.get("_error")]
    if not errs:
        continue
    print(f"Lote {s}–{e}: retentando {len(errs)} questão(ões)...")
    updated = {r["_idx"]: r for r in data}
    for r in errs:
        idx = r["_idx"]
        print(f"  idx={idx}...", end=" ")
        try:
            rev = rq.review_question(client, all_q[idx], idx, rq.DEFAULT_MODEL, rq.DEFAULT_EFFORT)
            rev["_t"]     = all_q[idx].get("t", "")
            rev["_idx"]   = idx
            rev["_diff"]  = all_q[idx].get("diff", "medium")
            rev["_cat"]   = all_q[idx].get("cat", "")
            rev["_refs"]  = all_q[idx].get("refs", [])
            rev["_model"] = rq.DEFAULT_MODEL
            updated[idx]  = rev
            print("OK")
        except Exception as ex:
            print(f"ERRO: {ex}")
        time.sleep(2)
    new_data = sorted(updated.values(), key=lambda x: x["_idx"])
    with open(fname, "w", encoding="utf-8") as f:
        json.dump(new_data, f, ensure_ascii=False, indent=2)
    subprocess.run(["python", "apply_reviews.py", fname])

print("\nRetentativa concluída.")
from google.colab import files
files.download("index.html")